In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Explore the first function
Import the `road_density` function to see an example of a reproducible function that calculates an infrastructure measure and merges with the corresponding country shapefile

In [1]:
from territorial_inequality.example_sunderland import road_density

In [2]:
zambia_roads = road_density("Zambia")
zambia_roads.head()

,Country,Cons_name,ISO,geometry,const_area_km2,road_length,length_km,road_density
0,Zambia,Mfuwe,ZMB,"POLYGON ((746087.63 -1332212.614, 745942.331 -...",19310.061053,79513.332035,79.513332,0.004118
1,Zambia,Kawambwa,ZMB,"POLYGON ((447937.343 -1099996.763, 447914.537 ...",3806.042306,76234.373996,76.234374,0.020030
2,Zambia,Nyimba,ZMB,"POLYGON ((661739.808 -1661906.008, 662825.846 ...",11035.485506,132537.567574,132.537568,0.012010
3,Zambia,Solwezi West,ZMB,"POLYGON ((118657.07 -1411749.529, 118826.84 -1...",16444.038250,228369.253212,228.369253,0.013888
4,Zambia,Kapiri Mposhi,ZMB,"POLYGON ((348932.876 -1638201.854, 349504.39 -...",16873.254809,108231.517411,108.231517,0.006414


We know that railroad density will follow the same calculation as road density, but using railway length instead of road length. This means that we can create a function that will take as input the layer (roads or railway) and create the corresponding measure.

In [3]:
from territorial_inequality.example_sunderland import transportation_density

In [4]:
zambia_railways = transportation_density(country = "Zambia", layer = "AFR_Infra_Transport_Rail")
zambia_railways.head()

,Country,Cons_name,ISO,geometry,const_area_km2,road_length,length_km,density
0,Zambia,Mfuwe,ZMB,"POLYGON ((746087.63 -1332212.614, 745942.331 -...",19310.061053,74981.149616,74.981150,0.003883
1,Zambia,Kawambwa,ZMB,"POLYGON ((447937.343 -1099996.763, 447914.537 ...",3806.042306,NaN,NaN,NaN
2,Zambia,Nyimba,ZMB,"POLYGON ((661739.808 -1661906.008, 662825.846 ...",11035.485506,NaN,NaN,NaN
3,Zambia,Solwezi West,ZMB,"POLYGON ((118657.07 -1411749.529, 118826.84 -1...",16444.038250,NaN,NaN,NaN
4,Zambia,Kapiri Mposhi,ZMB,"POLYGON ((348932.876 -1638201.854, 349504.39 -...",16873.254809,77721.548910,77.721549,0.004606


In [5]:
zambia_roads = transportation_density(country = "Zambia", layer = "AFR_Infra_Transport_Road")
zambia_roads.head()

,Country,Cons_name,ISO,geometry,const_area_km2,road_length,length_km,density
0,Zambia,Mfuwe,ZMB,"POLYGON ((746087.63 -1332212.614, 745942.331 -...",19310.061053,79513.332035,79.513332,0.004118
1,Zambia,Kawambwa,ZMB,"POLYGON ((447937.343 -1099996.763, 447914.537 ...",3806.042306,76234.373996,76.234374,0.020030
2,Zambia,Nyimba,ZMB,"POLYGON ((661739.808 -1661906.008, 662825.846 ...",11035.485506,132537.567574,132.537568,0.012010
3,Zambia,Solwezi West,ZMB,"POLYGON ((118657.07 -1411749.529, 118826.84 -1...",16444.038250,228369.253212,228.369253,0.013888
4,Zambia,Kapiri Mposhi,ZMB,"POLYGON ((348932.876 -1638201.854, 349504.39 -...",16873.254809,108231.517411,108.231517,0.006414


# Run a test
Run the test to ensure that length, constituency area, and density variables are all positive values.

In [10]:
!make test

pytest -v
============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-8.4.1, pluggy-1.5.0 -- /opt/anaconda3/envs/territorial-inequality/bin/python
cachedir: .pytest_cache
rootdir: /Users/sophiesunderland/Desktop/CMSE802F26
configfile: pyproject.toml
testpaths: territorial_inequality/tests
plugins: anyio-4.15.1
collected 3 items                                                              

territorial_inequality/tests/test_pytest.py::test_length_positive FAILED [ 33%]
territorial_inequality/tests/test_pytest.py::test_area_positive PASSED   [ 66%]
territorial_inequality/tests/test_pytest.py::test_density_positive FAILED [100%]

=================================== FAILURES ===================================
_____________________________ test_length_positive _____________________________

    def test_length_positive() -> None:
        """Template: replace this with tests for your own function."""
        result = transportat

The test results show there are some issues with the current data. All constituency area values are positive (these come from the shapefiles) but length and density values are not all positive.

It is possible that some values for length are NA because the current spatial join in the function keeps all consituency rows, even if they have NA values. This is worth investigating to edit the function to be able to handle true zero or NA values.

# Next steps

The `transportation_density` function works. Now we can extend this function to loop over all countries in the `Shapefiles` folder and saved the merged data.

This is what I am working on so far and testing before moving the function into my `example_sunderland` script. I am also planning to look into the data more to update my function to handle true zero vs NA values.

In [ ]:
# create output directory
output_dir = Path.home() / "Desktop" / "test" / "road_density_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

for country in df["Country"].dropna().unique():
    # create a GeoDataFrame for either roads/railways 
    transportation = gpd.read_file(df_main, layer = layer)
    # filter for the current country
    transportation = transportation[transportation["Country"] == country]
    # project the roads to a projected coordinate system for area calculations
    trans_proj = transportation.to_crs("ESRI:102022")
    
    # load shapefile for current country and project to the same coordinate system
    country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
    country_sf_proj = country_sf.to_crs("ESRI:102022")

    # intersect roads with constituencies 
    intersection = gpd.overlay(
        trans_proj,
        country_sf_proj,
        how="intersection")

    # create road variable length by summing the length of the roads in each constituency
    intersection["road_length"] = intersection.geometry.length.groupby(intersection["Cons_name"]).transform("sum")
    # transform from m into km
    intersection["length_km"] = intersection["road_length"] / 1000
    # select road length and constituency name columns
    intersection_sub = intersection[["road_length", "length_km", "Cons_name"]]

    # calculate constituency areas in km^2
    country_sf_proj["const_area_km2"] = country_sf_proj.geometry.area / 1_000_000

    # left join constituency areas with road lengths to get a combined dataframe
    country_combined = country_sf_proj.merge(intersection_sub, on="Cons_name", how="left")
    # calculate road density
    country_combined["density"] = (country_combined["length_km"] / country_combined["const_area_km2"])

    country_simple = country.lower().replace(" ", "_")
    output_path = output_dir / f"{country_simple}_combined.parquet"
    country_combined.to_parquet(output_path)
    print(f"Saved {country} -> {output_path}")